In [ ]:

"""
medvqa_pubmedclip_ban_accelerate.py

Patch gợi ý cho notebook Medical VQA classifier:
- Hỗ trợ 2 protocol answer vocabulary: train-only strict và train+eval để reproduce GitHub/paper.
- Dùng CrossEntropyLoss cho single-label VQA thay vì BCE one-hot nhân num_classes.
- Dùng answer_type từ dataset nếu có, không suy ra CLOSED = yes/no.
- Có auxiliary answer-type head + mask open/closed khi infer.
- Có is_howmany/answer_filter tương thích BAN/TMMPN.
- Có option trong TrainConfig để giữ/xóa count questions có answer không phải số lượng.
- Chạy được notebook 1 GPU hoặc 2 GPU bằng accelerate.notebook_launcher.

Cách chạy nhanh trong notebook:
    !pip install -q accelerate transformers datasets

    from medvqa_pubmedclip_ban_accelerate import TrainConfig, train_main
    from accelerate import notebook_launcher

    cfg = TrainConfig(
        dataset_name="flaviagiammarino/vqa-rad",
        model_name="flaviagiammarino/pubmed-clip-vit-base-patch32",
        output_dir="./medvqa_runs/vqa_rad_pubmedclip_ban",
        epochs=40,
        batch_size=16,
        lr_head=1e-4,
        lr_clip=1e-6,
        freeze_clip=True,
        answer_vocab_source="train_eval",  # repo/paper-compatible. Dùng "train" cho strict protocol.
        filter_invalid_count_answers=False,   # True: xóa count-question mà answer không phải số lượng.
        count_answer_max_num=None,            # None: chấp nhận mọi số; 10: giống BAN counting-only.
        mixed_precision="fp16",
    )

    # 1 GPU:
    train_main(cfg)

    # 2 GPU T4:
    # notebook_launcher(train_main, args=(cfg,), num_processes=2)
"""

from __future__ import annotations

import os
import re
import json
import math
import random
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.parametrizations import weight_norm
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor, get_cosine_schedule_with_warmup
from accelerate import Accelerator


# -----------------------
# Reproducibility
# -----------------------

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # CUDNN benchmark nhanh hơn, nhưng không deterministic tuyệt đối.
    torch.backends.cudnn.benchmark = True


# -----------------------
# Answer / question normalization
# -----------------------

MANUAL_MAP = {
    "none": "0", "zero": "0", "one": "1", "two": "2",
    "three": "3", "four": "4", "five": "5", "six": "6",
    "seven": "7", "eight": "8", "nine": "9", "ten": "10",
}
ARTICLES = {"a", "an", "the"}
PUNCT = [
    ";", "/", "[", "]", '"', "{", "}", "(", ")", "=",
    "+", "\\", "_", ">", "<", "@", "`", ",", "?", "!"
]
PERIOD_STRIP = re.compile(r"(?!<=\d)(\.)(?!\d)")
COMMA_STRIP = re.compile(r"(\d)(\,)(\d)")


def process_punctuation(text: str) -> str:
    text = str(text)
    out = text
    for p in PUNCT:
        if (p + " " in text) or (" " + p in text) or re.search(COMMA_STRIP, text):
            out = out.replace(p, "")
        else:
            out = out.replace(p, " ")
    # giữ dấu '-' trong vài cụm y khoa như "x-ray", rồi chuẩn hóa sau
    out = PERIOD_STRIP.sub("", out)
    return out


def process_digit_article(text: str) -> str:
    words = str(text).lower().split()
    return " ".join(MANUAL_MAP.get(w, w) for w in words if w not in ARTICLES)


def normalize_answer(answer: str) -> str:
    answer = str(answer).lower().strip()
    answer = answer.replace("? -yes/no", "")
    answer = answer.replace("? -open", "")
    answer = answer.replace("? - open", "")

    # Chuẩn hóa biến thể thường gặp trước khi tách dấu.
    answer = answer.replace("x ray", "xray").replace("x-ray", "xray")
    answer = answer.replace("ct scan", "ct").replace("mri scan", "mri")

    answer = process_punctuation(answer)
    answer = process_digit_article(answer)
    answer = answer.replace("\n", " ")
    answer = re.sub(r"\s+", " ", answer)
    return answer.strip()


def clean_question(question: str) -> str:
    question = str(question).strip()
    question = question.replace("? -yes/no", "")
    question = question.replace("? -open", "")
    question = question.replace("? - open", "")
    question = question.replace("x ray", "x-ray")
    question = re.sub(r"\s+", " ", question)
    return question


# -----------------------
# Counting question helper, same spirit as BAN/TMMPN
# -----------------------

def is_count_question(question: str) -> bool:
    """
    True nếu question là dạng hỏi số lượng/counting.
    Giữ cùng tinh thần với BAN/TMMPN: how many, number of, amount of, count of.
    """
    q = str(question).lower().strip()
    return (
        "how many" in q
        or ("number of" in q and "number of the" not in q)
        or "amount of" in q
        or "count of" in q
    )


def answer_filter(
    answer_obj_or_text: Any,
    label2ans: Optional[List[str]] = None,
    max_num: Optional[int] = 10,
) -> bool:
    """
    True nếu answer là số lượng hợp lệ.

    Hỗ trợ:
    - answer text: "2", "two", "2.0"
    - BAN/TMMPN target dict: {"labels": tensor/list}

    max_num:
    - 10: giống BAN counting-only gốc.
    - None: chấp nhận mọi số nguyên không âm, phù hợp hơn khi chỉ muốn lọc answer không phải số.
    """
    def _valid_text(ans_text: Any) -> bool:
        ans = normalize_answer(str(ans_text))

        if ans.isdigit():
            value = int(ans)
            return max_num is None or value <= max_num

        # Chấp nhận dạng "2.0" như một số đếm nguyên.
        try:
            value_float = float(ans)
            if value_float.is_integer() and value_float >= 0:
                value = int(value_float)
                return max_num is None or value <= max_num
        except ValueError:
            pass

        return False

    if isinstance(answer_obj_or_text, dict) and label2ans is not None:
        labels = answer_obj_or_text.get("labels", [])
        if torch.is_tensor(labels):
            labels = labels.detach().cpu().tolist()
        return any(_valid_text(label2ans[int(lab)]) for lab in labels)

    return _valid_text(answer_obj_or_text)


def is_valid_count_answer(answer: Any, max_num: Optional[int] = None) -> bool:
    """True nếu raw answer là số lượng hợp lệ. Mặc định không giới hạn <=10."""
    return answer_filter(answer, label2ans=None, max_num=max_num)


def is_invalid_count_answer_sample(sample: Dict[str, Any], max_num: Optional[int] = None) -> bool:
    """
    True nếu sample nên bị xóa theo rule:
    question hỏi số lượng nhưng answer không phải số lượng hợp lệ.
    """
    q = get_field(sample, ["question", "Question"], "")
    a = get_field(sample, ["answer", "Answer"], "")
    return is_count_question(q) and not is_valid_count_answer(a, max_num=max_num)


def is_howmany(
    question: str,
    answer: Optional[Any] = None,
    label2ans: Optional[List[str]] = None,
    max_num: Optional[int] = 10,
) -> bool:
    """
    Backward-compatible helper giống BAN/TMMPN.
    True khi question là counting và answer là số hợp lệ.
    Nếu answer=None thì chỉ kiểm tra question có phải counting hay không.
    """
    if not is_count_question(question):
        return False
    if answer is None:
        return True
    return answer_filter(answer, label2ans=label2ans, max_num=max_num)


# -----------------------
# Dataset helpers
# -----------------------

def get_field(sample: Dict[str, Any], names: Iterable[str], default: Any = None) -> Any:
    for n in names:
        if n in sample and sample[n] is not None:
            return sample[n]
    return default


def infer_answer_type(sample: Dict[str, Any], answer_norm: str) -> str:
    """
    Ưu tiên metadata answer_type nếu dataset có.
    Fallback yes/no chỉ dùng khi dataset không có metadata.
    Return: "closed" hoặc "open".
    """
    raw = get_field(sample, ["answer_type", "AnswerType", "answer_type_en", "type"], None)
    if raw is not None:
        raw_s = str(raw).lower()
        if "closed" in raw_s or "close" in raw_s or "yes/no" in raw_s:
            return "closed"
        if "open" in raw_s:
            return "open"

    # Một số HF dataset chỉ có question_type/content_type.
    qraw = get_field(sample, ["question_type", "content_type"], None)
    if qraw is not None:
        qraw_s = str(qraw).lower()
        if "closed" in qraw_s or "yes/no" in qraw_s:
            return "closed"
        if "open" in qraw_s:
            return "open"

    return "closed" if answer_norm in {"yes", "no"} else "open"


def build_answer_vocab(
    train_split,
    eval_split=None,
    min_freq: int = 1,
    max_answers: Optional[int] = None,
    source: str = "train_eval",
) -> Tuple[List[str], Dict[str, int], set]:
    """
    Build answer vocabulary theo protocol rõ ràng.

    source="train":
        Strict ML protocol. Candidate answers chỉ lấy từ train.
        Eval answers không nằm trong train vẫn giữ lại trong denominator và bị tính sai
        trong overall_acc. unseen_answer_ratio chỉ là diagnostic, không phải metric paper.

    source="train_eval" hoặc "all":
        Repo/paper-compatible protocol thường gặp ở Med-VQA codebases nhỏ.
        Candidate answers lấy từ train + eval split cố định để tránh lỗi label không tồn tại.
        Cách này dùng test labels để định nghĩa label space, nên cần ghi rõ trong báo cáo.
        Các answer chỉ xuất hiện ở eval vẫn gần như không học được nếu không có mask/semantic/generative head,
        nhưng ít nhất output dimension khớp với label2ans của repo.

    Không nên xóa eval samples mặc định: filter làm denominator nhỏ đi và accuracy bị inflated.
    """
    source = source.lower().strip()
    train_answers = [normalize_answer(x["answer"]) for x in train_split]
    train_answer_set = set(train_answers)

    if source in {"train", "strict"}:
        vocab_answers = train_answers
    elif source in {"train_eval", "all", "repo", "paper"}:
        if eval_split is None:
            raise ValueError("eval_split is required when source='train_eval'/'all'.")
        vocab_answers = train_answers + [normalize_answer(x["answer"]) for x in eval_split]
    else:
        raise ValueError(f"Unknown answer vocab source: {source}. Use 'train' or 'train_eval'.")

    cnt = Counter(vocab_answers)
    items = [(ans, freq) for ans, freq in cnt.items() if freq >= min_freq]
    # Sort ổn định: freq giảm dần, answer tăng dần.
    items = sorted(items, key=lambda x: (-x[1], x[0]))
    if max_answers is not None:
        items = items[:max_answers]
    label2ans = [ans for ans, _ in items]
    ans2label = {ans: i for i, ans in enumerate(label2ans)}
    return label2ans, ans2label, train_answer_set


def build_label_type_ids(train_split, ans2label: Dict[str, int]) -> torch.Tensor:
    """
    Mỗi answer label nhận type theo majority trong train.
    0=open, 1=closed.
    """
    votes: Dict[int, Counter] = defaultdict(Counter)
    for s in train_split:
        ans = normalize_answer(s["answer"])
        if ans not in ans2label:
            continue
        typ = infer_answer_type(s, ans)
        votes[ans2label[ans]][typ] += 1

    label_type = torch.zeros(len(ans2label), dtype=torch.long)
    for lab, c in votes.items():
        label_type[lab] = 1 if c["closed"] >= c["open"] else 0
    return label_type


class MedVQADataset(Dataset):
    def __init__(
        self,
        hf_split,
        ans2label: Dict[str, int],
        is_train: bool,
        train_answer_set: Optional[set] = None,
        filter_unseen_train_answers: bool = False,
        filter_invalid_count_answers: bool = False,
        count_answer_max_num: Optional[int] = None,
    ):
        self.ans2label = ans2label
        self.is_train = is_train
        self.train_answer_set = train_answer_set or set(ans2label.keys())
        self.count_answer_max_num = count_answer_max_num

        data = list(hf_split)
        self.original_len = len(data)

        # Diagnostic: đếm count-question mà answer không phải số lượng.
        self.invalid_count_answer_samples = [
            x for x in data if is_invalid_count_answer_sample(x, max_num=count_answer_max_num)
        ]
        self.num_invalid_count_answer_samples = len(self.invalid_count_answer_samples)

        if filter_invalid_count_answers:
            # Option mới: xóa sample hỏi số lượng nhưng answer không phải số lượng.
            # Cẩn thận khi bật cho eval/test: denominator thay đổi nên không so trực tiếp với paper.
            data = [
                x for x in data
                if not is_invalid_count_answer_sample(x, max_num=count_answer_max_num)
            ]

        if (not is_train) and filter_unseen_train_answers:
            # Chỉ dùng cho ablation/debug. Không dùng để so paper vì denominator thay đổi.
            data = [x for x in data if normalize_answer(x["answer"]) in self.train_answer_set]

        self.data = data
        self.filtered_len = len(self.data)
        self.num_filtered_samples = self.original_len - self.filtered_len

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        s = self.data[idx]
        image = s["image"].convert("RGB")
        q = clean_question(s["question"])
        ans = normalize_answer(s["answer"])

        label = self.ans2label.get(ans, -100)  # -100 nếu answer không nằm trong classifier vocab
        answer_type = infer_answer_type(s, ans)
        type_id = 1 if answer_type == "closed" else 0
        seen_in_train = ans in self.train_answer_set
        count_question = is_count_question(q)
        valid_count_answer = is_valid_count_answer(ans, max_num=self.count_answer_max_num)
        invalid_count_answer = count_question and not valid_count_answer

        return {
            "image": image,
            "question": q,
            "answer": ans,
            "label": torch.tensor(label, dtype=torch.long),
            "answer_type": torch.tensor(type_id, dtype=torch.long),
            # label == -100 nghĩa là classifier không có class này; thường chỉ xảy ra với source='train'.
            "is_unseen_answer": torch.tensor(label == -100, dtype=torch.bool),
            # Diagnostic quan trọng hơn: answer này có từng xuất hiện trong train không?
            "is_answer_seen_in_train": torch.tensor(seen_in_train, dtype=torch.bool),
            "is_count_question": torch.tensor(count_question, dtype=torch.bool),
            "is_invalid_count_answer": torch.tensor(invalid_count_answer, dtype=torch.bool),
        }


def make_collate_fn(processor: CLIPProcessor, max_length: int = 77):
    def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        images = [x["image"] for x in batch]
        questions = [x["question"] for x in batch]
        enc = processor(
            text=questions,
            images=images,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=max_length,
        )
        enc["labels"] = torch.stack([x["label"] for x in batch])
        enc["answer_type"] = torch.stack([x["answer_type"] for x in batch])
        enc["is_unseen_answer"] = torch.stack([x["is_unseen_answer"] for x in batch])
        enc["is_answer_seen_in_train"] = torch.stack([x["is_answer_seen_in_train"] for x in batch])
        enc["is_count_question"] = torch.stack([x["is_count_question"] for x in batch])
        enc["is_invalid_count_answer"] = torch.stack([x["is_invalid_count_answer"] for x in batch])
        enc["answers_text"] = [x["answer"] for x in batch]
        return enc
    return collate_fn


# -----------------------
# BAN modules
# -----------------------

class FCNet(nn.Module):
    def __init__(self, dims: List[int], act: str = "ReLU", dropout: float = 0.2):
        super().__init__()
        layers = []
        for i in range(len(dims) - 1):
            layers.append(weight_norm(nn.Linear(dims[i], dims[i + 1],bias=False), name="weight", dim=None))
            if act:
                layers.append(getattr(nn, act)())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        self.main = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.main(x)


class BCNet(nn.Module):
    def __init__(self, v_dim: int, q_dim: int, h_dim: int, h_out: Optional[int] = None,
                 dropout: Tuple[float, float] = (0.2, 0.5), k: int = 3):
        super().__init__()
        self.h_dim = h_dim
        self.h_out = h_out
        self.k = k
        self.v_net = FCNet([v_dim, h_dim * k], act="ReLU", dropout=dropout[0])
        self.q_net = FCNet([q_dim, h_dim * k], act="ReLU", dropout=dropout[0])
        self.dropout = nn.Dropout(dropout[1])
        self.p_net = nn.AvgPool1d(k, stride=k) if k > 1 else None

        if h_out is not None:
            self.h_mat = nn.Parameter(torch.randn(1, h_out, 1, h_dim * k) * 0.01)
            self.h_bias = nn.Parameter(torch.zeros(1, h_out, 1, 1))

    def forward(self, v: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
        v_ = self.dropout(self.v_net(v))
        q_ = self.q_net(q)
        if self.h_out is None:
            return torch.einsum("bvk,bqk->bvqk", v_, q_)
        return torch.einsum("ghyk,bvk,bqk->bhvq", self.h_mat, v_, q_) + self.h_bias

    def forward_with_weights(self, v: torch.Tensor, q: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
        v_ = self.v_net(v)
        q_ = self.q_net(q)
        logits = torch.einsum("bvk,bvq,bqk->bk", v_, w, q_)
        if self.k > 1:
            logits = self.p_net(logits.unsqueeze(1)).squeeze(1) * self.k
        return logits


class BiAttention(nn.Module):
    def __init__(self, v_dim: int, q_dim: int, h_dim: int, glimpse: int = 4):
        super().__init__()
        self.glimpse = glimpse
        self.logits = BCNet(v_dim, q_dim, h_dim, h_out=glimpse, dropout=(0.2, 0.5), k=3)

    def forward_all(self, v: torch.Tensor, q: torch.Tensor, q_mask: Optional[torch.Tensor] = None):
        B, V, Q = v.size(0), v.size(1), q.size(1)
        logits = self.logits(v, q)
        if q_mask is not None:
            mask = q_mask[:, None, None, :].bool()
            logits = logits.masked_fill(~mask, -1e4)
        att = F.softmax(logits.reshape(B, self.glimpse, V * Q), dim=-1).reshape(B, self.glimpse, V, Q)
        return att, logits


class SimpleClassifier(nn.Module):
    def __init__(self, in_dim: int, hid_dim: int, out_dim: int, dropout: float = 0.5):
        super().__init__()
        self.main = nn.Sequential(
            weight_norm(nn.Linear(in_dim, hid_dim,bias=False), name="weight", dim=None),
            nn.ReLU(),
            nn.Dropout(dropout),
            weight_norm(nn.Linear(hid_dim, out_dim,bias=False), name="weight", dim=None),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.main(x)


class PubMedCLIPBANFixed(nn.Module):
    def __init__(
        self,
        clip_model: CLIPModel,
        num_answers: int,
        num_hid: int = 512,
        glimpse: int = 4,
        freeze_clip: bool = True,
    ):
        super().__init__()
        self.clip = clip_model
        self.freeze_clip = freeze_clip
        self.v_dim = clip_model.config.vision_config.hidden_size
        self.q_dim = clip_model.config.text_config.hidden_size
        self.glimpse = glimpse

        if freeze_clip:
            for p in self.clip.parameters():
                p.requires_grad = False

        self.v_att = BiAttention(self.v_dim, self.q_dim, num_hid, glimpse=glimpse)
        self.b_net = nn.ModuleList([
            BCNet(self.v_dim, self.q_dim, num_hid, h_out=None, dropout=(0.2, 0.5), k=3)
            for _ in range(glimpse)
        ])
        self.q_prj = nn.ModuleList([
            FCNet([num_hid, self.q_dim], act="ReLU", dropout=0.2)
            for _ in range(glimpse)
        ])

        self.answer_classifier = SimpleClassifier(self.q_dim, num_hid * 2, num_answers, dropout=0.5)
        self.type_classifier = SimpleClassifier(self.q_dim, num_hid, 2, dropout=0.3)

    def train(self, mode: bool = True):
        super().train(mode)
        if self.freeze_clip:
            self.clip.eval()
        return self

    def encode_tokens(self, pixel_values, input_ids, attention_mask):
        grad_enabled = any(p.requires_grad for p in self.clip.parameters())
        with torch.set_grad_enabled(grad_enabled):
            vision_outputs = self.clip.vision_model(pixel_values=pixel_values, return_dict=True)
            text_outputs = self.clip.text_model(
                input_ids=input_ids, attention_mask=attention_mask, return_dict=True
            )
        # ViT: bỏ CLS token; ResNet CLIP có thể cần chỉnh lại nếu dùng RN backend.
        visual_tokens = vision_outputs.last_hidden_state[:, 1:, :]
        text_tokens = text_outputs.last_hidden_state
        return visual_tokens, text_tokens

    def forward(self, pixel_values, input_ids, attention_mask):
        v, q = self.encode_tokens(pixel_values, input_ids, attention_mask)
        att, _ = self.v_att.forward_all(v, q, q_mask=attention_mask)

        for g in range(self.glimpse):
            b_emb = self.b_net[g].forward_with_weights(v, q, att[:, g])
            q = q + self.q_prj[g](b_emb).unsqueeze(1)

        mask = attention_mask.unsqueeze(-1).float()
        q_pooled = (q * mask).sum(1) / mask.sum(1).clamp_min(1.0)

        answer_logits = self.answer_classifier(q_pooled)
        type_logits = self.type_classifier(q_pooled)
        return answer_logits, type_logits, att


def mask_logits_by_predicted_type(
    logits: torch.Tensor,
    type_logits: torch.Tensor,
    label_type_ids: torch.Tensor,
    mask_value: float = -1e4,
) -> torch.Tensor:
    """
    label_type_ids: [num_answers], 0=open, 1=closed
    type_logits -> predicted type per sample.
    """
    pred_type = type_logits.argmax(dim=-1)  # [B]
    allowed = label_type_ids.unsqueeze(0).to(logits.device) == pred_type.unsqueeze(1)
    return logits.masked_fill(~allowed, mask_value)


# -----------------------
# Training
# -----------------------

@dataclass
class TrainConfig:
    dataset_name: str = "flaviagiammarino/vqa-rad"
    model_name: str = "flaviagiammarino/pubmed-clip-vit-base-patch32"
    output_dir: str = "./medvqa_runs/pubmedclip_ban"
    seed: int = 42
    epochs: int = 40
    batch_size: int = 16
    eval_batch_size: int = 64
    num_workers: int = 2
    max_length: int = 77
    num_hid: int = 512
    glimpse: int = 4
    freeze_clip: bool = True
    lr_head: float = 1e-4
    lr_clip: float = 1e-6
    weight_decay: float = 1e-4
    warmup_ratio: float = 0.05
    type_loss_weight: float = 0.2
    class_weight_power: float = 0.5   # inverse freq^power
    mixed_precision: str = "fp16"     # "no", "fp16", "bf16"
    grad_clip: float = 1.0
    min_answer_freq: int = 1
    max_answers: Optional[int] = None
    # "train_eval" giống nhiều repo Med-VQA nhỏ: label space biết trước từ split cố định.
    # "train" là strict protocol: không dùng eval labels để tạo candidate answers.
    answer_vocab_source: str = "train_eval"
    # Chỉ bật để debug/ablation. Không bật khi so paper vì làm accuracy cao giả tạo.
    filter_eval_unseen_train_answers: bool = False

    # Option mới cho counting questions:
    # False: giữ nguyên dataset, chỉ log invalid_count_answer_ratio. Khuyến nghị khi so paper.
    # True: xóa sample có question hỏi số lượng nhưng answer không phải số lượng. Dùng cho clean/debug/ablation.
    filter_invalid_count_answers: bool = False
    # None: answer số lượng nào cũng hợp lệ, ví dụ 0, 2, 15.
    # 10: giống BAN counting-only gốc, chỉ nhận số <= 10.
    count_answer_max_num: Optional[int] = None

    eval_with_type_mask: bool = True


def build_class_weights(train_split, ans2label: Dict[str, int], power: float = 0.5) -> torch.Tensor:
    cnt = torch.ones(len(ans2label), dtype=torch.float32)  # smoothing
    for s in train_split:
        ans = normalize_answer(s["answer"])
        if ans in ans2label:
            cnt[ans2label[ans]] += 1.0
    weights = 1.0 / torch.pow(cnt, power)
    weights = weights / weights.mean()
    return weights


@torch.no_grad()
def evaluate(model, loader, accelerator: Accelerator, label_type_ids: torch.Tensor, cfg: TrainConfig) -> Dict[str, float]:
    model.eval()
    device = accelerator.device

    totals = torch.zeros(18, device=device, dtype=torch.float64)
    # [0 correct_all, 1 total_all,
    #  2 open_correct_all, 3 open_total_all, 4 closed_correct_all, 5 closed_total_all,
    #  6 not_in_classifier_vocab, 7 total_for_vocab,
    #  8 seen_train_correct, 9 seen_train_total,
    # 10 unseen_train_correct,11 unseen_train_total,
    # 12 answerable_vocab_correct,13 answerable_vocab_total,
    # 14 count_correct,15 count_total,16 invalid_count_answer_total,17 valid_count_answer_total]

    for batch in loader:
        answer_logits, type_logits, _ = model(
            pixel_values=batch["pixel_values"],
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        if cfg.eval_with_type_mask:
            answer_logits = mask_logits_by_predicted_type(answer_logits, type_logits, label_type_ids.to(device))

        labels = batch["labels"]
        ans_type = batch["answer_type"]
        not_in_classifier_vocab = batch["is_unseen_answer"]
        seen_train = batch["is_answer_seen_in_train"]
        answerable = labels.ne(-100)

        preds = answer_logits.argmax(dim=-1)
        correct = preds.eq(labels) & answerable

        open_mask = ans_type.eq(0)
        closed_mask = ans_type.eq(1)
        unseen_train = ~seen_train
        count_mask = batch["is_count_question"]
        invalid_count_answer = batch["is_invalid_count_answer"]

        # Overall paper-style denominator: mọi eval sample đều được tính; answer không có trong vocab là sai.
        totals[0] += correct.sum()
        totals[1] += labels.numel()
        totals[2] += (correct & open_mask).sum()
        totals[3] += open_mask.sum()
        totals[4] += (correct & closed_mask).sum()
        totals[5] += closed_mask.sum()

        totals[6] += not_in_classifier_vocab.sum()
        totals[7] += labels.numel()

        totals[8] += (correct & seen_train).sum()
        totals[9] += seen_train.sum()
        totals[10] += (correct & unseen_train).sum()
        totals[11] += unseen_train.sum()

        totals[12] += correct.sum()
        totals[13] += answerable.sum()

        totals[14] += (correct & count_mask).sum()
        totals[15] += count_mask.sum()
        totals[16] += invalid_count_answer.sum()
        totals[17] += (count_mask & ~invalid_count_answer).sum()

    totals = accelerator.reduce(totals, reduction="sum")
    if accelerator.num_processes > 1:
        accelerator.wait_for_everyone()

    def div(a, b):
        return (a / max(b, 1.0)).item()

    return {
        # Dùng metric này để so paper: denominator = toàn bộ eval split, không filter.
        "overall_acc": div(totals[0], totals[1]),
        "open_acc": div(totals[2], totals[3]),
        "closed_acc": div(totals[4], totals[5]),
        # Diagnostic, không phải metric paper.
        "not_in_classifier_vocab_ratio": div(totals[6], totals[7]),
        "unseen_train_answer_ratio": div(totals[11], totals[1]),
        "seen_train_acc": div(totals[8], totals[9]),
        "unseen_train_acc": div(totals[10], totals[11]),
        # Giữ lại để debug strict source='train': accuracy nếu bỏ qua sample không có class.
        "overall_acc_answerable_vocab_only": div(totals[12], totals[13]),
        "eval_total": int(totals[1].item()),
        "eval_answerable_vocab_total": int(totals[13].item()),
        "eval_seen_train_total": int(totals[9].item()),
        "eval_unseen_train_total": int(totals[11].item()),
        "count_question_acc": div(totals[14], totals[15]),
        "count_question_ratio": div(totals[15], totals[1]),
        "invalid_count_answer_ratio": div(totals[16], totals[1]),
        "eval_count_question_total": int(totals[15].item()),
        "eval_invalid_count_answer_total": int(totals[16].item()),
    }


def make_loaders(cfg: TrainConfig, processor: CLIPProcessor):
    ds = load_dataset(cfg.dataset_name)

    train_split = ds["train"]
    # Nếu dataset có validation thì dùng validation; nếu không, dùng test để theo notebook cũ.
    eval_name = "validation" if "validation" in ds else "test"
    eval_split = ds[eval_name]

    label2ans, ans2label, train_answer_set = build_answer_vocab(
        train_split,
        eval_split=eval_split,
        min_freq=cfg.min_answer_freq,
        max_answers=cfg.max_answers,
        source=cfg.answer_vocab_source,
    )
    label_type_ids = build_label_type_ids(train_split, ans2label)
    class_weights = build_class_weights(train_split, ans2label, power=cfg.class_weight_power)

    train_set = MedVQADataset(
        train_split,
        ans2label,
        is_train=True,
        train_answer_set=train_answer_set,
        filter_invalid_count_answers=cfg.filter_invalid_count_answers,
        count_answer_max_num=cfg.count_answer_max_num,
    )
    eval_set = MedVQADataset(
        eval_split,
        ans2label,
        is_train=False,
        train_answer_set=train_answer_set,
        filter_unseen_train_answers=cfg.filter_eval_unseen_train_answers,
        filter_invalid_count_answers=cfg.filter_invalid_count_answers,
        count_answer_max_num=cfg.count_answer_max_num,
    )
    collate_fn = make_collate_fn(processor, max_length=cfg.max_length)

    train_loader = DataLoader(
        train_set,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=cfg.num_workers,
        pin_memory=True,
        persistent_workers=cfg.num_workers > 0,
    )
    eval_loader = DataLoader(
        eval_set,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=cfg.num_workers,
        pin_memory=True,
        persistent_workers=cfg.num_workers > 0,
    )

    filter_stats = {
        "train_original_len": train_set.original_len,
        "train_filtered_len": train_set.filtered_len,
        "train_invalid_count_answer_samples": train_set.num_invalid_count_answer_samples,
        "eval_original_len": eval_set.original_len,
        "eval_filtered_len": eval_set.filtered_len,
        "eval_invalid_count_answer_samples": eval_set.num_invalid_count_answer_samples,
        "filter_invalid_count_answers": cfg.filter_invalid_count_answers,
        "count_answer_max_num": cfg.count_answer_max_num,
    }

    return train_loader, eval_loader, label2ans, ans2label, label_type_ids, class_weights, eval_name, train_answer_set, filter_stats


def train_main(cfg: TrainConfig):
    seed_everything(cfg.seed)

    accelerator = Accelerator(mixed_precision=cfg.mixed_precision)
    if accelerator.is_main_process:
        os.makedirs(cfg.output_dir, exist_ok=True)
        with open(os.path.join(cfg.output_dir, "config.json"), "w", encoding="utf-8") as f:
            json.dump(asdict(cfg), f, indent=2, ensure_ascii=False)

    processor = CLIPProcessor.from_pretrained(cfg.model_name)
    (
        train_loader,
        eval_loader,
        label2ans,
        ans2label,
        label_type_ids,
        class_weights,
        eval_name,
        train_answer_set,
        filter_stats,
    ) = make_loaders(cfg, processor)

    if accelerator.is_main_process:
        print(
            "[count-filter] "
            f"enabled={filter_stats['filter_invalid_count_answers']} | "
            f"max_num={filter_stats['count_answer_max_num']} | "
            f"train {filter_stats['train_original_len']} -> {filter_stats['train_filtered_len']} "
            f"(invalid_count={filter_stats['train_invalid_count_answer_samples']}) | "
            f"{eval_name} {filter_stats['eval_original_len']} -> {filter_stats['eval_filtered_len']} "
            f"(invalid_count={filter_stats['eval_invalid_count_answer_samples']})"
        )

    clip_model = CLIPModel.from_pretrained(cfg.model_name)
    model = PubMedCLIPBANFixed(
        clip_model=clip_model,
        num_answers=len(label2ans),
        num_hid=cfg.num_hid,
        glimpse=cfg.glimpse,
        freeze_clip=cfg.freeze_clip,
    )

    # Hai nhóm LR: head/BAN lớn hơn, CLIP nhỏ hơn.
    head_params, clip_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("clip."):
            clip_params.append(p)
        else:
            head_params.append(p)

    optimizer = torch.optim.AdamW(
        [
            {"params": head_params, "lr": cfg.lr_head},
            {"params": clip_params, "lr": cfg.lr_clip},
        ],
        weight_decay=cfg.weight_decay,
    )

    total_steps = cfg.epochs * math.ceil(len(train_loader) / max(accelerator.num_processes, 1))
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    model, optimizer, train_loader, eval_loader, scheduler = accelerator.prepare(
        model, optimizer, train_loader, eval_loader, scheduler
    )

    class_weights = class_weights.to(accelerator.device)
    label_type_ids = label_type_ids.to(accelerator.device)

    best = -1.0
    best_path = os.path.join(cfg.output_dir, "best_model.pt")

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        loss_meter = 0.0
        step_count = 0

        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            answer_logits, type_logits, _ = model(
                pixel_values=batch["pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )

            answer_loss = F.cross_entropy(
                answer_logits,
                batch["labels"],
                weight=class_weights,
                ignore_index=-100,
            )
            type_loss = F.cross_entropy(type_logits, batch["answer_type"])
            loss = answer_loss + cfg.type_loss_weight * type_loss

            accelerator.backward(loss)
            if cfg.grad_clip and cfg.grad_clip > 0:
                accelerator.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            scheduler.step()

            loss_meter += accelerator.gather_for_metrics(loss.detach()).mean().item()
            step_count += 1

        metrics = evaluate(model, eval_loader, accelerator, label_type_ids, cfg)

        if accelerator.is_main_process:
            score = metrics["overall_acc"]
            if score > best:
                best = score
                unwrapped = accelerator.unwrap_model(model)
                torch.save(
                    {
                        "model": unwrapped.state_dict(),
                        "label2ans": label2ans,
                        "ans2label": ans2label,
                        "label_type_ids": label_type_ids.detach().cpu(),
                        "train_answer_set": sorted(list(train_answer_set)),
                        "answer_vocab_source": cfg.answer_vocab_source,
                        "filter_stats": filter_stats,
                        "config": asdict(cfg),
                    },
                    best_path,
                )

            print(
                f"Epoch {epoch:03d}/{cfg.epochs} | "
                f"loss={loss_meter / max(step_count, 1):.4f} | "
                f"{eval_name}_overall={metrics['overall_acc']:.4f} | "
                f"open={metrics['open_acc']:.4f} | "
                f"closed={metrics['closed_acc']:.4f} | "
                f"unseen_train_ratio={metrics['unseen_train_answer_ratio']:.3f} | "
                f"seen_train_acc={metrics['seen_train_acc']:.4f} | "
                f"unseen_train_acc={metrics['unseen_train_acc']:.4f} | "
                f"count_q_ratio={metrics['count_question_ratio']:.3f} | "
                f"invalid_count_ratio={metrics['invalid_count_answer_ratio']:.3f} | "
                f"best={best:.4f}"
            )

    accelerator.wait_for_everyone()
    return best_path


# if __name__ == "__main__":
#     train_main(TrainConfig())